# 7.03 SQL Database 도구 — `SQLDatabase` · `SQLDatabaseToolkit` · `create_sql_agent`

관계형 DB 를 에이전트에 노출하는 세 레벨의 API를 다룬다.

1. **`SQLDatabase`** — 접속·스키마 조회·쿼리 실행의 얇은 래퍼 (원시 레벨)
2. **`SQLDatabaseToolkit`** — 위 래퍼를 4개 도구로 묶어 제공 (list tables · get schema · run query · query checker)
3. **`create_sql_agent`** — 위 toolkit 을 자동으로 물려 즉시 `text-to-SQL` 에이전트를 만드는 팩토리

이 노트북은 repo 에 이미 있는 `05_advanced/Chinook.db` (SQLite) 를 재사용한다 — 키·원격 서버 불필요.

## 학습 목표

- `SQLDatabase.from_uri(...)` 로 접속·스키마 파악
- Toolkit 이 내보내는 **4개 도구**(`sql_db_list_tables`, `sql_db_schema`, `sql_db_query`, `sql_db_query_checker`) 와 역할
- `create_sql_agent(agent_type="tool-calling")` 로 3줄 text-to-SQL 에이전트
- 민감한 쓰기 쿼리 (`DELETE`, `UPDATE`) 는 **HITL** 로 감싸는 패턴

## 언제 쓰나

- 사내 정형 DB 에 자연어 질의 — BI 대시보드 대체
- RAG 컨텍스트 보강용 **관계형 fact lookup**
- 에이전트에 "특정 사용자 주문 조회" 같은 정밀 조회 능력 부여

## 7.03.1 환경 설정

SQLite 파일 기반 — 네트워크 DB 나 크레덴셜 불필요. `langchain-community` 의 SQL 모듈만 있으면 된다.

In [ ]:
# !pip install -U langchain langchain-community SQLAlchemy

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_community.utilities import SQLDatabase

DB_PATH = Path("/Users/baem1n/Workspace/langchain-langgraph-deepagents-notebooks/05_advanced/Chinook.db")
assert DB_PATH.exists(), f"Chinook.db not found at {DB_PATH}"

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
print("dialect :", db.dialect)
print("tables  :", db.get_usable_table_names()[:8], "...")

## 7.03.2 기본 사용 — `SQLDatabase` 직접 조회

가장 얇은 레벨. `.run(sql)` 은 실행 결과를 **문자열로 직렬화**해 반환 (에이전트 출력에 그대로 꽂기 좋은 포맷). `.get_table_info([...])` 은 `CREATE TABLE` DDL + 샘플 3행을 함께 반환 — text-to-SQL 프롬프트의 표준 컨텍스트.

In [ ]:
print("--- 샘플 쿼리 실행 ---")
print(db.run("SELECT Name FROM Artist LIMIT 3"))

print("\n--- Album 테이블 스키마 ---")
print(db.get_table_info(["Album"])[:600])

## 7.03.3 `SQLDatabaseToolkit` — 에이전트용 4개 도구

`SQLDatabaseToolkit(db=..., llm=...)` 이 내보내는 도구는 네 가지이며, 역할은 text-to-SQL 파이프라인의 단계 하나씩에 해당한다.

| 도구 | 역할 |
|------|------|
| `sql_db_list_tables` | 사용 가능한 테이블 이름 나열 |
| `sql_db_schema` | 특정 테이블의 DDL + 샘플 행 |
| `sql_db_query` | SQL 실행 후 결과 반환 |
| `sql_db_query_checker` | **LLM 호출** 로 SQL 문법·안전성 사전 검증 |

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
    from langchain_openai import ChatOpenAI

    toolkit = SQLDatabaseToolkit(db=db, llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0))
    tools = toolkit.get_tools()
    print("tool 수 :", len(tools))
    for t in tools:
        print("-", t.name, "|", t.description[:80].replace("\n", " "))
else:
    print("OPENAI_API_KEY 없음 → 섹션 7.03.3, 7.03.4 스킵. 섹션 7.03.2 는 키 없이 동작 확인됨.")
    tools = None

## 7.03.4 `create_sql_agent` — 3줄 text-to-SQL 에이전트

`create_sql_agent` 는 **LLM + toolkit** 을 받아 `AgentExecutor` 를 만든다. `agent_type="tool-calling"` 이 v1 권장값.

> import 경로 드리프트 주의: v1 에서는 `langchain.agents.create_sql_agent` 가 **제거**됨. 현재 공식 경로는 `langchain_community.agent_toolkits.sql.base.create_sql_agent`.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain_community.agent_toolkits.sql.base import create_sql_agent

    sql_agent = create_sql_agent(
        llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
        toolkit=toolkit,
        agent_type="tool-calling",
        verbose=False,
    )
    res = sql_agent.invoke({"input": "어느 장르(Genre)의 트랙이 가장 많지? 상위 3개만 알려줘."})
    print(res["output"])
else:
    print("OPENAI_API_KEY 없음 → 스킵.")

## 7.03.5 v1 스타일 — `create_agent` + `toolkit.get_tools()`

LangChain v1 의 통일 에이전트 팩토리 `create_agent` 에 toolkit 도구를 그대로 주입할 수도 있다. `create_sql_agent` (legacy) 대신 이 경로가 나머지 프로젝트 스타일과 일관된다.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_agent

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=toolkit.get_tools(),
        system_prompt=(
            "너는 SQLite 분석가다. 먼저 sql_db_list_tables 로 테이블을 확인하고, "
            "필요한 테이블의 sql_db_schema 를 읽은 뒤, sql_db_query 로 질의를 실행해 답한다. "
            "절대 DROP/DELETE/UPDATE 같은 쓰기 쿼리는 실행하지 마라."
        ),
    )
    res = agent.invoke({"messages": [{"role": "user", "content": "고객(Customer) 총 몇 명이야?"}]})
    print(res["messages"][-1].content)
else:
    print("OPENAI_API_KEY 없음 → 스킵.")

## 7.03.6 주요 옵션 · 비교 표

### `SQLDatabase.from_uri` 파라미터

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `sample_rows_in_table_info` | 3 | `get_table_info` 에 포함되는 샘플 행 수. 프롬프트 토큰 비용에 직결 |
| `include_tables` | None | 허용 테이블 화이트리스트 |
| `ignore_tables` | None | 블랙리스트 (민감 테이블 숨기기) |
| `view_support` | False | DB 뷰도 테이블처럼 노출할지 |
| `custom_table_info` | None | 테이블명 → DDL 문자열 커스텀 오버라이드 |

### `create_sql_agent` agent_type

| 값 | 설명 |
|---|------|
| `tool-calling` (**권장**) | OpenAI/Anthropic 등 tool-calling 지원 모델용. 가장 안정적 |
| `openai-tools` | 과거 OpenAI 전용 — tool-calling 과 동등 |
| `zero-shot-react-description` | 레거시 ReAct 프롬프트 — 최신 모델에는 불필요 |

### DB 방언

| Dialect | URI 예 | 비고 |
|---------|--------|------|
| SQLite | `sqlite:///path/to.db` | 파일 기반, 로컬 프로토타입 |
| PostgreSQL | `postgresql+psycopg://user:pw@host/db` | 프로덕션 표준 |
| MySQL | `mysql+pymysql://user:pw@host/db` | |
| Snowflake | `snowflake://user:pw@account/db/schema` | |
| BigQuery | `bigquery://project/dataset` | `sqlalchemy-bigquery` 필요 |

## 7.03.7 (선택) HITL 결합 — 쓰기 쿼리 승인

SQL 에이전트의 가장 큰 리스크는 **모델이 `DELETE`/`UPDATE`/`DROP` 을 실행**하는 것. 두 가지 방어층을 권장한다.

1. **DB 레이어**: 읽기 전용 사용자 계정으로 접속 (권한 최소화)
2. **에이전트 레이어**: `HumanInTheLoopMiddleware` 로 `sql_db_query` 를 승인 게이트

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=toolkit.get_tools(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"sql_db_query": {"allowed_decisions": ["approve", "edit", "reject"]}},
    )],
)
```

`edit` 으로 모델이 쓴 SQL 을 검수 후 실행. `LIMIT` 이 누락된 집계 쿼리를 사람이 수정하는 용도로도 유용. 전체 resume 흐름은 `02_langchain/07_hitl_and_runtime` 참고.

## 체크리스트

- [ ] 프로덕션에서는 **읽기 전용 DB 계정** 으로 `from_uri` — 에이전트가 실수해도 DML 불가
- [ ] `include_tables` / `ignore_tables` 로 **민감 테이블** (직원 연봉, 결제 토큰 등) 노출 차단
- [ ] 큰 테이블은 `sample_rows_in_table_info=1` 로 프롬프트 토큰 절약
- [ ] `sql_db_query_checker` 는 **매번 LLM 호출** 이라 비용·지연 증가 — 신뢰 가능 모델이면 제거 가능
- [ ] DB 방언에 따라 `LIMIT` vs `TOP` vs `FETCH FIRST N ROWS ONLY` — 모델이 틀리면 시스템 프롬프트에 방언 명시

## 다음

- `02_langchain/07_hitl_and_runtime` — HITL resume 전체 흐름
- `05_advanced/06_agentic_sql.ipynb` — SQL 에이전트 튜닝 (few-shot, 스키마 요약 등)

## 참고

- LangChain SQL 가이드: https://python.langchain.com/docs/tutorials/sql_qa/
- Chinook DB 스키마: https://github.com/lerocha/chinook-database